# 09-10 Colab Sparse Autoencoder And Text-To-Brain Pipeline

Run-all Colab notebook for Stage 1 sparse autoencoder and Stage 2 text-to-brain projection variants. It clones the repo and copies generated data from Drive automatically.



## 1. Runtime And Drive Mount

This notebook is designed for Colab Run all. It clones the repository into `/content/neurovlm`, copies your generated data from Google Drive into the clone, and writes run outputs back to Drive.


In [ ]:
import os, sys, json, time, platform, subprocess, shutil
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else 'No GPU detected by nvidia-smi. Use Runtime -> Change runtime type -> GPU.')

from google.colab import drive
drive.mount('/content/drive')


## 2. Clone Or Pull Repository

In [ ]:
REPO_URL = 'https://github.com/neurovlm/neurovlm.git'
REPO_BRANCH = 'neurovlm_gnn'
REPO_DIR = Path('/content/neurovlm')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=False)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repo ready at', REPO_DIR)
print(subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())


## 3. Install Dependencies

In [ ]:
%pip install -q -e /content/neurovlm
%pip install -q torch pandas numpy nibabel nilearn pyyaml tqdm safetensors transformers adapters pyarrow matplotlib scikit-learn


## 4. Link Generated Drive Data Into The Clone

This assumes your generated data lives under `/content/drive/MyDrive/neurovlm/` with `atlas_free_multipositive/cache/` and `data_ale_3dcnn/ale_caches/`. The notebook links those folders into `/content/neurovlm/` and rewrites JSONL `tensor_path` values to the actual ALE cache location.


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')
DRIVE_PROJECT_DIR = DRIVE_ROOT / 'neurovlm'

if not DRIVE_PROJECT_DIR.exists():
    raise FileNotFoundError(f'Missing Drive project directory: {DRIVE_PROJECT_DIR}')

essential_drive_paths = [
    DRIVE_PROJECT_DIR / 'atlas_free_multipositive/cache/unified_jsonl/splits/train.jsonl',
    DRIVE_PROJECT_DIR / 'atlas_free_multipositive/cache/unified_jsonl/splits/val.jsonl',
    DRIVE_PROJECT_DIR / 'atlas_free_multipositive/cache/unified_jsonl/text_registry.jsonl',
    DRIVE_PROJECT_DIR / 'atlas_free_multipositive/cache/text_embeddings/specter_text_cache.pt',
    DRIVE_PROJECT_DIR / 'data_ale_3dcnn/ale_caches/atlas_free_4mm_fwhm9_crop_float16.pt',
]
missing_drive = [str(p) for p in essential_drive_paths if not p.exists()]
if missing_drive:
    raise FileNotFoundError('Missing required Drive files: ' + repr(missing_drive))

print('Using generated data from:', DRIVE_PROJECT_DIR)

# Link generated data into the cloned runtime repo. This avoids copying the big ALE cache.
links = {
    REPO_DIR / 'atlas_free_multipositive/cache':
        DRIVE_PROJECT_DIR / 'atlas_free_multipositive/cache',
    REPO_DIR / 'data_ale_3dcnn/ale_caches':
        DRIVE_PROJECT_DIR / 'data_ale_3dcnn/ale_caches',
}

for dst, src in links.items():
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        print('exists:', dst)
    else:
        dst.symlink_to(src, target_is_directory=True)
        print('linked:', dst, '->', src)

# Rewrite JSONL paths to the actual ALE cache location you have on Drive.
# This changes tensor_path from atlas_free_multipositive/data/ale_caches/... to data_ale_3dcnn/ale_caches/...
from atlas_free_multipositive.data_building.rewrite_jsonl_paths import rewrite_jsonl_paths

old_cache_dir = 'atlas_free_multipositive/data/ale_caches'
new_cache_dir = 'data_ale_3dcnn/ale_caches'
for split_jsonl in [
    REPO_DIR / 'atlas_free_multipositive/cache/unified_jsonl/splits/train.jsonl',
    REPO_DIR / 'atlas_free_multipositive/cache/unified_jsonl/splits/val.jsonl',
]:
    changed = rewrite_jsonl_paths(split_jsonl, old=old_cache_dir, new=new_cache_dir)
    print('rewrote', changed, 'tensor/nifti path values in', split_jsonl)

DRIVE_RUNS_DIR = DRIVE_PROJECT_DIR / 'atlas_free_multipositive/outputs/runs'
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
print('Drive run outputs:', DRIVE_RUNS_DIR)


## 5. Verify Local Training Files And JSONL References


In [ ]:
import json

required_local = [
    REPO_DIR / 'atlas_free_multipositive/cache/unified_jsonl/splits/train.jsonl',
    REPO_DIR / 'atlas_free_multipositive/cache/unified_jsonl/splits/val.jsonl',
    REPO_DIR / 'atlas_free_multipositive/cache/unified_jsonl/text_registry.jsonl',
    REPO_DIR / 'atlas_free_multipositive/cache/text_embeddings/specter_text_cache.pt',
    REPO_DIR / 'data_ale_3dcnn/ale_caches/atlas_free_4mm_fwhm9_crop_float16.pt',
]

missing = []
for p in required_local:
    ok = p.exists()
    print(f'{str(ok):5}  {p}')
    if not ok:
        missing.append(str(p))

# Validate actual tensor_path/nifti_path references used by the Dataset.
referenced = set()
for split in ['train', 'val']:
    split_path = REPO_DIR / f'atlas_free_multipositive/cache/unified_jsonl/splits/{split}.jsonl'
    with split_path.open() as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            for key in ['tensor_path', 'nifti_path']:
                value = row.get(key)
                if value:
                    referenced.add(value)

for value in sorted(referenced):
    p = Path(value)
    p = p if p.is_absolute() else REPO_DIR / p
    if not p.exists():
        missing.append(str(p))

print('checked JSONL-referenced map/tensor paths:', len(referenced))
if missing:
    raise FileNotFoundError('Missing files after Drive link/path rewrite: ' + repr(missing[:20]))
print('All required files and JSONL-referenced paths are available.')


## 6. Imports And Stage Settings

In [ ]:
import copy, yaml, torch
from atlas_free_multipositive.training.train_autoencoder_sparse import train_from_config as train_sparse_autoencoder
from atlas_free_multipositive.training.train_text_to_brain import train_from_config as train_text_to_brain
BASE_RUN_DIR = DRIVE_RUNS_DIR / time.strftime('colab_generation_%Y%m%d_%H%M%S')
BASE_RUN_DIR.mkdir(parents=True, exist_ok=True)

# G4 / ~96GB GPU defaults. These are intentionally much larger than the smoke-test configs.
# Stage 1 is usually CPU/I/O/metric bound, so use larger batches and fewer metric batches.
AE_BATCH_SIZE = 128
AE_EPOCHS = 100
TEXT_DECODER_BATCH_SIZE = 256
TEXT_DECODER_EPOCHS = 100
BASE_CHANNELS = 48
NUM_BLOCKS = 4
LATENT_DIM = 384
MAX_VAL_BATCHES = None
NUM_WORKERS = 8
VAL_INTERVAL = 5
TRAIN_METRIC_BATCHES = 8
VAL_METRIC_BATCHES = 16
METRICS_DEVICE = 'cuda'
INCLUDE_VOXEL_AUROC = False
USE_AMP = True

print('BASE_RUN_DIR =', BASE_RUN_DIR)
print('AE_BATCH_SIZE =', AE_BATCH_SIZE, 'TEXT_DECODER_BATCH_SIZE =', TEXT_DECODER_BATCH_SIZE)
print('NUM_WORKERS =', NUM_WORKERS, 'VAL_INTERVAL =', VAL_INTERVAL, 'USE_AMP =', USE_AMP)
print('BASE_CHANNELS =', BASE_CHANNELS, 'NUM_BLOCKS =', NUM_BLOCKS)


## 7. Stage 1 - Sparse CNN Autoencoder

In [ ]:
ae_cfg = yaml.safe_load(Path('atlas_free_multipositive/configs/autoencoder_sparse_config.yaml').read_text())
ae_cfg.update({
    'output_dir': str(BASE_RUN_DIR / 'stage1_sparse_autoencoder'),
    'checkpoint_dir': str(BASE_RUN_DIR / 'stage1_sparse_autoencoder' / 'checkpoints'),
    'device': 'auto',
    'epochs': AE_EPOCHS,
    'batch_size': AE_BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'pin_memory': True,
    'persistent_workers': True,
    'prefetch_factor': 4,
    'max_train_batches': None,
    'max_val_batches': MAX_VAL_BATCHES,
    'amp': USE_AMP,
    'cudnn_benchmark': True,
    'val_interval': VAL_INTERVAL,
    'compute_train_metrics': True,
    'train_metric_batches': TRAIN_METRIC_BATCHES,
    'val_metric_batches': VAL_METRIC_BATCHES,
    'metrics_device': METRICS_DEVICE,
    'include_voxel_auroc': INCLUDE_VOXEL_AUROC,
    'lr': 3e-4,
    'model': {'latent_dim': LATENT_DIM, 'base_channels': BASE_CHANNELS, 'num_blocks': NUM_BLOCKS},
})
print('Stage 1 config:', {k: ae_cfg[k] for k in ['epochs', 'batch_size', 'lr', 'model']})
ae_result = train_sparse_autoencoder(ae_cfg)
AE_CHECKPOINT = Path(ae_cfg['checkpoint_dir']) / 'best_generation_top5_dice.pt'
if not AE_CHECKPOINT.exists():
    AE_CHECKPOINT = Path(ae_cfg['checkpoint_dir']) / 'last.pt'
print('AE_CHECKPOINT =', AE_CHECKPOINT)


## 8. Stage 2A - Text-To-Brain Projection, Random Init

In [ ]:
base_t2b_cfg = yaml.safe_load(Path('atlas_free_multipositive/configs/text_to_brain_config.yaml').read_text())
t2b_random = copy.deepcopy(base_t2b_cfg)
t2b_random.update({
    'autoencoder_checkpoint': str(AE_CHECKPOINT),
    'text_projection_init': 'random',
    'output_dir': str(BASE_RUN_DIR / 'stage2_text_to_brain_random'),
    'checkpoint_dir': str(BASE_RUN_DIR / 'stage2_text_to_brain_random' / 'checkpoints'),
    'device': 'auto',
    'epochs': TEXT_DECODER_EPOCHS,
    'batch_size': TEXT_DECODER_BATCH_SIZE,
    'max_train_batches': None,
    'max_val_batches': MAX_VAL_BATCHES,
    'lr': 1e-4,
    'positive_texts_per_map': 1,
    'model': {'latent_dim': LATENT_DIM, 'base_channels': BASE_CHANNELS, 'num_blocks': NUM_BLOCKS},
})
print('Stage 2 random config:', {k: t2b_random[k] for k in ['epochs', 'batch_size', 'lr', 'text_projection_init', 'model']})
random_result = train_text_to_brain(t2b_random)
RANDOM_T2B_CHECKPOINT = Path(t2b_random['checkpoint_dir']) / 'best_generation_top5_dice.pt'
if not RANDOM_T2B_CHECKPOINT.exists():
    RANDOM_T2B_CHECKPOINT = Path(t2b_random['checkpoint_dir']) / 'last.pt'
print('RANDOM_T2B_CHECKPOINT =', RANDOM_T2B_CHECKPOINT)


## 9. Stage 2B - Text-To-Brain Projection, Pretrained Text InfoNCE Init

In [ ]:
t2b_pretrained = copy.deepcopy(base_t2b_cfg)
t2b_pretrained.update({
    'autoencoder_checkpoint': str(AE_CHECKPOINT),
    'text_projection_init': 'pretrained_text_infonce',
    'output_dir': str(BASE_RUN_DIR / 'stage2_text_to_brain_pretrained_text_infonce'),
    'checkpoint_dir': str(BASE_RUN_DIR / 'stage2_text_to_brain_pretrained_text_infonce' / 'checkpoints'),
    'device': 'auto',
    'epochs': TEXT_DECODER_EPOCHS,
    'batch_size': TEXT_DECODER_BATCH_SIZE,
    'max_train_batches': None,
    'max_val_batches': MAX_VAL_BATCHES,
    'lr': 1e-4,
    'positive_texts_per_map': 1,
    'model': {'latent_dim': LATENT_DIM, 'base_channels': BASE_CHANNELS, 'num_blocks': NUM_BLOCKS},
})
print('Stage 2 pretrained config:', {k: t2b_pretrained[k] for k in ['epochs', 'batch_size', 'lr', 'text_projection_init', 'model']})
pretrained_result = train_text_to_brain(t2b_pretrained)
PRETRAINED_T2B_CHECKPOINT = Path(t2b_pretrained['checkpoint_dir']) / 'best_generation_top5_dice.pt'
if not PRETRAINED_T2B_CHECKPOINT.exists():
    PRETRAINED_T2B_CHECKPOINT = Path(t2b_pretrained['checkpoint_dir']) / 'last.pt'
print('PRETRAINED_T2B_CHECKPOINT =', PRETRAINED_T2B_CHECKPOINT)


## 10. Save Manifest

In [ ]:
manifest = {'base_run_dir': str(BASE_RUN_DIR), 'autoencoder_checkpoint': str(AE_CHECKPOINT), 'random_text_to_brain_checkpoint': str(RANDOM_T2B_CHECKPOINT), 'pretrained_text_infonce_text_to_brain_checkpoint': str(PRETRAINED_T2B_CHECKPOINT), 'stage1_result': ae_result, 'stage2_random_result': random_result, 'stage2_pretrained_result': pretrained_result}
json.dump(manifest, open(BASE_RUN_DIR / 'generation_stage_manifest.json', 'w'), indent=2)
print(json.dumps(manifest, indent=2))
